# LAB 5: 웨어하우스와 컨텍스트

👉 이 실습에서는 가상 웨어하우스에 대해 더 자세히 살펴보고, 그 크기(컴퓨팅 용량)를 조정하는 방법과 함께 Snowsight에서 비용 정보를 확인할 수 있는 옵션들을 알아볼 것입니다.

시작하기에 앞서, 이 실습 전반에서 사용할 **컨텍스트 정보**를 가져오겠습니다. 

- **Start** 버튼을 클릭하여 이 노트북을 활성화하세요.

- 다음 Python 셀을 실행하세요.


#### :warning: 이 노트북에 대해 새 세션이 시작될 때마다, 후속 셀에서 사용할 '변수'를 구성하기 위해 아래 셀을 다시 실행해야 합니다. :warning:


In [ ]:
import streamlit as st
from snowflake.snowpark.context import get_active_session
session = get_active_session()
user = session.get_current_user().strip('"')
your_db = user + '_DB'
print('현재 CONTEXT 정보:')
print('---------------------------------')
print(session)
print('현재 USER는 ' + user)

## Snowflake에서 웨어하우스란 무엇인가요? 📓 

### Snowflake의 '웨어하우스' 정의하기

- 데이터 작업을 오래 해온 사람들은 '데이터 웨어하우스'라는 용어가 특별한 데이터 구조 집합을 가리킨다고 생각할 수 있지만, Snowflake에서 웨어하우스는 데이터를 저장하지 않습니다.

- Snowflake에서 웨어하우스는 '컴퓨팅 리소스'입니다. 데이터 처리를 수행하는 데 사용됩니다. 

- Snowflake에서 웨어하우스를 생성한다는 것은 이러한 '리소스'를 정의하는 것입니다.

### Scaling Up(수직 확장)과 Down(수직 축소) 

- 웨어하우스의 크기를 변경하면 클러스터 내 서버 수가 달라집니다. 

- 기존 웨어하우스의 크기를 변경하는 것을 'Scaling Up(수직 확장)' 또는 'Scaling Down(수직 축소)'이라고 합니다.

### Scaling In(수평 축소)와 Out(수평 확장) 

- 멀티 클러스터 또는 탄력적 웨어하우징(Enterprise Edition 이상)을 사용할 수 있는 경우, 웨어하우스는 수요 증가에 맞춰 Scaling Out(수평 확장)할 수 있습니다.

- 멀티 클러스터 Scaling Out(수평 확장)이 발생하면, 수요가 있는 동안 클러스터가 추가되었다가 수요가 줄어들면 다시 제거(snap back)됩니다.

- 초기 클러스터에 포함된 서버 수에 따라 웨어하우스가 클러스터를 추가하여 Scaling Out(수평 확장)될 때 각 클러스터에 포함되는 서버 수가 결정됩니다. 


## 단지 가능하다고 해서... 📓 

### ...반드시 해야 하는 뜻은 아닙니다. 📓 

Snowflake에서는 단 몇 초 만에 엄청난 컴퓨팅 파워를 활용할 수 있습니다. 특히 **대형** 컴퓨팅 파워가 필요한 대규모 작업을 수행할 때 이 기능이 가능하다는 점을 알려드리고자 합니다. 

하지만 대부분의 쿼리는 막대한 컴퓨팅 파워를 **필요로 하지 않는다**는 점도 함께 알려드립니다. 

사실 Snowflake는 항상 Extra-Small(XS) 웨어하우스에서 시작하고, 그럴 만한 명확한 이유가 있을 때만 Scaling Up(수직 확장)할 것을 권장합니다. XS 웨어하우스는 시간당 1크레딧을 사용합니다. 가장 큰 웨어하우스는 Snowpark-optimized 6XL로, 무려 시간당 768크레딧을 사용합니다. 즉, 분당 12크레딧 이상입니다. 

이번 워크숍에서는, 특별히 크기 S를 사용하라고 안내하는 경우를 제외하고는 웨어하우스를 XS로 유지하세요. 

실제 업무 환경에서 Snowflake를 사용할 경우, 웨어하우스 구성을 감독하는 담당자가 따로 있는 경우가 많습니다. 웨어하우스를 과도하게 크게 설정하는 것은 월별 청구서에서 큰 비용 충격을 초래하는 가장 쉬운 실수이므로, 대부분의 경우 XS와 S 웨어하우스를 사용하는 데 익숙해지고, 충분한 검토 후에만 Scaling Up(수직 확장)하는 것이 좋습니다.

Snowflake는 각 계정마다 비용을 감독하고, 최적의 웨어하우스 크기를 선택하는 방법과 탄력성 설정 구성에 관한 전문 지식을 갖춘 담당자를 둘 것을 권장합니다. 이러한 비용 관리자는 웨어하우스 크기 변경이 추가 비용을 감당할 만큼의 시간 절감 효과를 가져오는지, 혹은 비용과 절감 효과가 서로 상쇄되는지를 계산할 수 있어야 합니다. 


## 가상 웨어하우스 크기 조정 🥋

Snowflake 가상 웨어하우스 크기는 Snowsight를 통해 또는 프로그래밍 방식으로 조정할 수 있습니다. 할당된 가상 웨어하우스를 통해 이러한 옵션을 살펴보겠습니다.

### Snowsight를 사용하여 가상 웨어하우스를 'Scaling Up(수직 확장)'합니다.

새 브라우저 탭 또는 윈도우에서 Snowsight Warehouses 페이지를 엽니다.

1. 메인 메뉴에서 Notebook 왼쪽의 **컴퓨팅** 아이콘 위에 마우스를 올리세요.

1. 대화 상자에 나타나는 **Warehouses** 링크를 마우스 오른쪽 버튼으로 클릭하고 원하는 대상(새 탭 또는 새 윈도우)을 선택하세요.

1. 이 작업을 수행하면 Snowflake Notebook 페이지를 중단하지 않은 상태로 유지하면서, 지침 화면으로 쉽게 다시 전환할 수 있습니다.

![웨어하우스 열기 페이지(이미지)](https://edu-cdev-images.s3.us-west-2.amazonaws.com/ob/ob_admin_menu_1_v2.png)

1. 본인의 동물 이름이 붙은 웨어하우스 라인을 찾으세요.

1. 해당 라인 오른쪽의 줄임표를 클릭한 후 **Edit** 옵션을 선택하세요.

![웨어하우스 편집 대화 상자 열기(이미지)](https://edu-cdev-images.s3.us-west-2.amazonaws.com/ob/ob_warehouse_edit_1.png)

1. **Size** 드롭다운을 클릭하세요.

1. **Small 2 credits/hour** 옵션을 선택하세요.

1. 파란색 **Save Warehouse** 옵션을 선택하세요.

![웨어하우스 크기 조정(이미지)](https://edu-cdev-images.s3.us-west-2.amazonaws.com/ob/ob_warehouse_edit_2.png)


### 코드를 사용하여 가상 웨어하우스 크기를 확인하세요. 🥋

`SHOW` 명령을 사용하면 현재 크기를 포함한 가상 웨어하우스의 다양한 세부 정보를 확인할 수 있습니다.

다음 명령을 실행하고, 웨어하우스의 'size' 컬럼에 표시된 값을 기록하세요.


In [ ]:
SHOW warehouses like '{{user}}_wh';

### 코드를 사용하여 가상 웨어하우스 크기를 조정하세요. 🥋

가상 웨어하우스에 대해 이러한 작업을 수행할 권한이 있다면, `ALTER` 구문을 사용하여 다양한 속성을 변경하거나, 실행 중인 쿼리를 취소하고, 웨어하우스를 일시 중단(stop) 또는 재개(start)할 수 있습니다.

`ALTER` 구문을 실행하여, 이전 연습에서 Snowsight를 통해 Scaling Up(수직 확장)했던 가상 웨어하우스를 원래 사이즈로 되돌리세요.


In [ ]:
ALTER warehouse {{user}}_wh SET warehouse_size=XSMALL;

### `SHOW` 명령을 사용하여 웨어하우스 크기 변경이 성공적으로 이루어졌는지 확인하세요. 🎯 

- 아래 SQL 셀에 정보(현재 크기 포함)를 표시하기 위해 `SHOW` 명령을 **작성**하고 실행하세요.

💡 **힌트**: 다음 셀에는 코드가 제공되지 않았지만, 이전에 이 노트북에서 실행한 명령어를 참고하여 작성할 수 있습니다.


In [ ]:
-- 여기에 SHOW 명령어를 작성하고 실행하여 현재 가상 웨어하우스 크기를 확인하세요

### 예상치 못한 상황으로부터 보호하기 📓 

Snowflake는 비용을 모니터링하고 제어하는 다양한 방법을 제공하므로, 누군가 실수를 해도 최대한 빨리 알 수 있습니다. 플랫폼 관리자는 일반적으로 Snowflake 계정에서 이러한 기능을 구성하고 관리합니다. 주요 옵션은 다음과 같습니다.

- [Budgets](https://docs.snowflake.com/ko/user-guide/budgets)
- [Resource Monitors](https://docs.snowflake.com/ko/user-guide/resource-monitors)
- [Alerts and Notifications](https://docs.snowflake.com/ko/guides-overview-alerts)


### 비용 정보 보기 📓

Snowflake는 플랫폼 내 지출을 추적할 수 있도록 다양한 특성과 기능을 제공합니다. 이 정보를 얻기 위해 **SNOWFLAKE** 데이터베이스의 **ACCOUNT_USAGE** 및 **ORGANIZATION_USAGE** 스키마에 있는 뷰를 대상으로 쿼리를 작성할 수 있습니다. 더 간단하게는 Snowsight를 사용하여 과거 비용을 탐색할 수도 있습니다. Snowsight를 사용하면 시각적 대시보드를 통해 비용 정보를 빠르고 쉽게 확인할 수 있습니다. 

기본적으로 비용 및 사용량 데이터에 액세스할 수 있는 권한은 계정 관리자(즉, `ACCOUNTADMIN` 역할을 가진 사용자)에게만 있지만, 이 계정에서는 해당 권한이 `(animal)_LEARNER_RL`에 부여되어 있습니다. 

계정 수준 비용을 확인하는 방법
- Snowsight의 왼쪽 메인 메뉴에서 **Admin** » **Cost Management**를 선택한 후, 이를 새 브라우저 탭이나 윈도우에서 여세요.

![비용 관리 액세스(이미지)](https://edu-cdev-images.s3.us-west-2.amazonaws.com/ob/ob_cost_management_link_1_v2.png)

### Account Overview

- 안내 메시지가 표시되면, 사용량 데이터를 확인하는 데 사용할 웨어하우스를 선택하세요.
- **Account Overview** 페이지는 Snowflake 사용 비용을 한눈에 파악할 수 있는 인사이트를 제공하며, 지출 최적화를 위한 출발점이 될 수 있습니다.

![Account Overview(이미지)](https://edu-cdev-images.s3.us-west-2.amazonaws.com/ob/ob_account_overview_ui_v2.png)

### Consumption

- **Consumption** 페이지를 사용하여 일별, 주별 또는 월별 Snowflake 사용 전체 비용을 상세하게 분석할 수 있습니다.

![Account Overview(이미지)](https://edu-cdev-images.s3.us-west-2.amazonaws.com/ob/ob_consumption_ui_v2.png)


**참고**: 이 페이지들에는 다양한 선택형 하위 메뉴와 클릭 가능한 구성 요소들이 포함되어 있어, 요구 사항에 따라 비용을 여러 가지 방식으로 상세 분석하고, 필터링하고, 정렬할 수 있습니다.


## Lesson 5 챌린지 실습 🎯 

### 계정 수준 비용을 확인하세요.

#### Cost Management > Account Overview 페이지 🎯 

![Account Overview(이미지)](https://edu-cdev-images.s3.us-west-2.amazonaws.com/ob/ob_account_overview_ui_small_v2.png)

Snowsight에서 직접 이 페이지를 살펴보고 다음 질문에 답하세요.

- 이 계정의 지난 28일 동안 **average daily credits**은 얼마입니까?
- 가장 큰 데이터베이스는 무엇입니까?
- 이 기간 동안 가장 비용이 많이 든 쿼리를 실행한 사용자는 누구입니까?


#### Cost Management > Consumption 페이지 🎯 

![Account Overview(이미지)](https://edu-cdev-images.s3.us-west-2.amazonaws.com/ob/ob_consumption_ui_small_v2.png)

Snowsight에서 직접 이 페이지를 살펴보고 다음 질문에 답하세요.

- 지난 28일 동안 가장 많은 크레딧이 사용된 날은 언제인가요?
- 이 기간 동안 AI 서비스가 크레딧을 사용했나요?


## 컨텍스트 설정 📓 

**Does not exist** 오류가 자주 발생하며, 항상 잘못된 ROLE을 사용했기 때문만은 아닙니다.

때때로 'Does not exist' 오류는 다음과 같은 다른 이유로 발생할 수 있습니다.

- **잘못된 위치에** 무언가를 **생성함** - 예를 들어 ROOT_DEPTH 테이블을 FRUITS 스키마에 넣은 경우

- **잘못된 위치에서 조회함** - 예를 들어 `SELECT * FROM ROOT_DEPTH;`를 조회하려는데 데이터베이스 컨텍스트가 `SNOWFLAKE_SAMPLE_DATA.PUBLIC`로 설정된 경우

- **오타**가 있음 - 예: `SELECT * FROM GARENPLNT.VEGGIES.ROOT_DEPTH;`

물론 해당 항목을 아예 **생성하지 않았을 가능성**도 있습니다. 그러니 그 가능성도 꼭 확인하세요.

이제 다음 사항을 이해하고 있어야 합니다.

- USER(사용자)와 ROLE(역할)의 차이점 

- USER(사용자)와 ACCOUNT(계정)의 차이점

- DEFAULT ROLE(기본 역할)과 Snowsight 홈페이지의 CURRENT ROLE(현재 역할), 노트북에서 설정한 ROLE(역할) 간의 차이점

- Snowflake Notebook에서 프로그래밍 방식으로 역할을 변경하는 방법

이 내용을 모른다면 아래 질문에 답하기 어려울 수 있습니다. 어려움을 겪는다면, 해당 수업을 다시 복습하세요.


## 지식 테스트 :mag_right:

아래의 대화형 퀴즈 문제를 통해 이해도를 확인해 보세요. 각 `RUN_THIS_QUIZ_QUESTION_` 셀에는 Snowflake 기능과 관련된 객관식 문제를 제시하는 Streamlit 위젯이 포함되어 있습니다.  

**지침:**  
1. 노트북 셀 위에 커서를 올려 추가 컨트롤을 표시하세요.
1. 각 퀴즈 셀 오른쪽의 ▶️ **Play 버튼**을 클릭하여 실행하세요.  
1. 제공된 옵션에서 답을 선택하세요.  
1. 다음으로 넘어가기 전에 피드백을 검토하세요.

💡 **참고:** 궁금하시면 셀을 확장하여 코드를 볼 수 있지만, 필수는 아닙니다. 이 퀴즈들은 필수 사항이 아닙니다. 배운 내용을 복습하며 연습할 기회를 제공하기 위한 것입니다.  


In [ ]:
import streamlit as st
st.divider()
question = "Snowflake Warehouse에 대한 참인 진술은 무엇인가요?"
options = ["아래 선택을 고르세요...",
           "A) Snowflake Warehouses는 조직의 데이터 하위 섹션을 보유하는 데이터 마트와 같습니다", 
           "B) 기능적으로, Snowflake Warehouse는 노트북 CPU보다는 스프레드시트 워크북과 더 비슷합니다", 
           "C) Snowflake Warehouses는 쿼리를 실행하고 데이터를 로드하며 기타 작업을 수행하기 위한 컴퓨팅 파워를 제공합니다",
           "D) Snowflake Warehouses는 Kimball과 Inmon 데이터 웨어하우징 이론에 기반한 구조에 데이터를 저장하도록 설계되었습니다"]

user_answer = st.radio(question, options, index=0)
if user_answer:
    if user_answer == "아래 선택을 고르세요...":
        ''
    else:
        answer = '81444c835e49add037d24a1be365a841'
        # 옵션 문자 (A, B, C, 또는 D)를 추출하세요
        selected_option = user_answer.split(')')[0] + ')'
        response = session.sql(f"call common_db.resources.quiz_temp('{answer}', '{user_answer}', 'False')").collect()
        if response:
            value = response[0]['QUIZ_TEMP']
        st.write(f"{selected_option} {value}")

In [ ]:
st.divider()
question = "Scaling OUT/BACK과 Scaling UP/DOWN은 쉽게 혼동됩니다. Snowflake 웨어하우스 스케일링에 대한 올바른 설명은 무엇입니까?"
options = ["아래 선택을 고르세요...",
           "A) XS 웨어하우스를 M으로 변경하는 것은 Scaling UP의 예입니다", 
           "B) XS 웨어하우스를 M으로 변경하는 것은 Scaling OUT의 예입니다", 
           "C) XS 웨어하우스가 증가된 작업 부하를 처리하기 위해 자동으로 클러스터를 추가하는 것은 Scaling Up의 예입니다",
           "D) XS 웨어하우스가 증가된 작업 부하를 처리하기 위해 자동으로 클러스터를 추가하는 것은 Scaling Down의 예입니다"]

user_answer = st.radio(question, options, index=0)
if user_answer:
    if user_answer == "아래 선택을 고르세요...":
        ''
    else:
        answer = 'c4ea1b1f7dc14dc7ebedf7dc9d371362'
        # 옵션 문자 (A, B, C, 또는 D)를 추출하세요
        selected_option = user_answer.split(')')[0] + ')'
        response = session.sql(f"call common_db.resources.quiz_temp('{answer}', '{user_answer}', 'False')").collect()
        if response:
            value = response[0]['QUIZ_TEMP']
        st.write(f"{selected_option} {value}")

In [ ]:
st.divider()
question = "Warehouses는 수동으로 UP 또는 DOWN으로 규모 조정할 수 있습니다. 또한 자동으로 Scale Out하도록 설정할 수 있습니다. Scaling Out의 반대는 무엇인가요?"
options = ["아래 선택을 고르세요...",
           "A) De-Scaling", 
           "B) Scaling Back", 
           "C) Shedding Clusters",
           "D) Scaling DOWN"]

user_answer = st.radio(question, options, index=0)
if user_answer:
    if user_answer == "아래 선택을 고르세요...":
        ''
    else:
        answer = 'd32360b1231ec658361bec6b720bd70b'
        # 옵션 문자 (A, B, C, 또는 D)를 추출하세요
        selected_option = user_answer.split(')')[0] + ')'
        response = session.sql(f"call common_db.resources.quiz_temp('{answer}', '{user_answer}', 'False')").collect()
        if response:
            value = response[0]['QUIZ_TEMP']
        st.write(f"{selected_option} {value}")

In [ ]:
st.divider()
question = "웨어하우스 규모조정은 서버와 클러스터 모두와 관련이 있습니다. 웨어하우스 크기조정 및 규모조정에서 서버와 클러스터의 역할에 대한 올바른 설명은 무엇입니까?"
options = ["아래 선택을 고르세요...",
           "A) 서버는 클러스터의 \"그룹\"을 의미합니다", 
           "B) M 크기의 웨어하우스(Scale Out되지 않은 경우)는 XS 크기의 웨어하우스보다 더 많은 클러스터를 가지고 있습니다", 
           "C) 서버는 여러 클러스터를 보유할 수 있습니다",
           "D) 클러스터는 여러 서버를 포함할 수 있습니다"]

user_answer = st.radio(question, options, index=0)
if user_answer:
    if user_answer == "아래 선택을 고르세요...": 
        ''
    else:
        answer = '88bd96557d74389e4b31c49bda96d6a4'
        # 옵션 문자 (A, B, C, 또는 D)를 추출하세요
        selected_option = user_answer.split(')')[0] + ')'
        response = session.sql(f"call common_db.resources.quiz_temp('{answer}', '{user_answer}', 'False')").collect()
        if response:
            value = response[0]['QUIZ_TEMP']
        st.write(f"{selected_option} {value}")

In [ ]:
st.divider()
question = "기존 웨어하우스를 언제 Scale Up 해야 합니까?"
options = ["아래 선택을 고르세요...",
           "A) 지루할 때 재미있어 보일 때", 
           "B) Scaling Up의 영향을 신중히 고려했을 때: 시간당 사용되는 크레딧 증가", 
           "C) 자유분방한 동료가 보수적인 동료보다 더 크게 소리를 지르며 그렇게 하라고 할 때"]

user_answer = st.radio(question, options, index=0)
if user_answer:
    if user_answer == "아래 선택을 고르세요...": 
        ''
    else:
        answer = '7adf4a847127fc42564abd23d6dbbebb'
        # 옵션 문자 (A, B, C, 또는 D)를 추출하세요
        selected_option = user_answer.split(')')[0] + ')'
        response = session.sql(f"call common_db.resources.quiz_temp('{answer}', '{user_answer}', 'False')").collect()
        if response:
            value = response[0]['QUIZ_TEMP']
        st.write(f"{selected_option} {value}")

## 다음 단계

실습 단계를 완료하고 **지식 테스트** 질문에 정답을 입력하셨다면, Snowflake 강사의 안내에 따라 다음 Notebook으로 진행하세요.
